In [3]:
import os

# mount drive if colab
from google.colab import drive
drive.mount('/content/gdrive')
os.chdir("/content/gdrive/MyDrive/Soil_Moisture/Local_Training")

Mounted at /content/gdrive


In [4]:
import os
import numpy as np
import tensorflow as tf


OUTPUT_NAME = "Fine_Tuning_Osiris" 
Grandvillers_path = '/content/gdrive/My Drive/Grandvillers/Grandvillers'
ROOT_DIR = "/content/gdrive/MyDrive/Soil_Moisture/dataset_training" # Si on Colab
OSIRIS_DIR = os.path.join(ROOT_DIR, "Osiris_unified") # Si on Colab

drive_dir = os.path.join("/content/gdrive/MyDrive/Soil_Moisture/outputs", OUTPUT_NAME) # Si on Colab
RESULTS_CSV_PATH = os.path.join(drive_dir, "results.csv") # Si on Colab

# Surcharge des chemins lus par config.py (via variables d'environnement).
# À définir AVANT l'import de Fcn_Training (cellule suivante).
os.environ["ROOT_DIR"] = ROOT_DIR
os.environ["OSIRIS_DIR_UNIFIED"] = OSIRIS_DIR
os.environ["OUTPUT_NAME"] = OUTPUT_NAME
os.environ["DRIVE_DIR"] = drive_dir


# FOLDER_NAME = "Fine_Tuning" 

# ROOT_DIR = "/home/theo/Dataset" # Si on local machine

# OSIRIS_DIR = os.path.join(ROOT_DIR, "Osiris_dataset")

# Grandvillers_path = os.path.join(ROOT_DIR, "Grandvillers_data")

# drive_dir = os.path.join("/home/theo/Documents", FOLDER_NAME, "outputs") # Si on local machine

# RESULTS_CSV_PATH = os.path.join(drive_dir, "results.csv") # Si on local machine

# ============================================================
# Configuration
# ============================================================

FOLDER_ISMN = "station_depth_csv"
MASTER_CSV_PATH = os.path.join(ROOT_DIR, FOLDER_ISMN, "Soil_Properties_Master.csv")

base_path = os.path.join(ROOT_DIR, FOLDER_ISMN, "depth")

FULL_DENSE_h = [ "soil_moisture", "ET0", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN",
              "VPD", "T_RANGE",
              "RAIN_CUM_3D", "RAIN_CUM_7D", "RAIN_CUM_14D",
              "doy_sin", "doy_cos"]

FULL_DENSE = [  "ET0", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN",
              "VPD", "T_RANGE",
              "RAIN_CUM_3D", "RAIN_CUM_7D", "RAIN_CUM_14D",
              "doy_sin", "doy_cos"]

FULL_SOIL  = [ "clay", "silt", "bulk", "sand", "dem", "ksat_m_1km", "dem_slope", "dem_aspect", "dem_twi"]

FULL_SPARSE = ["S2_B2", "S2_B3", "S2_B4", "S2_B5", "S2_B6", "S2_B7",
               "S2_B8", "S2_B8A", "S2_B11", "S2_B12",
               "S2_NDVI", "S2_NDWI", "S2_SAVI", "S2_MNDWI", "S2_NBR",
               "S1_VV", "S1_VH", "S1_angle",
               "S1_VV_over_VH", "S1_VH_over_VV",
               "HLS_B2", "HLS_B3", "HLS_B4", "HLS_B5", 
                     "HLS_B6", "HLS_B7", "HLS_B9", "HLS_B10", "HLS_B11", "HLS_NDVI"]

FULL_SPARSE_S1 = ["S1_VV", "S1_VH", "S1_angle",
                   "S1_VV_over_VH", "S1_VH_over_VV"]

FULL_SPARSE_S2 = ["S2_B2", "S2_B3", "S2_B4", "S2_B5", "S2_B6", "S2_B7",
               "S2_B8", "S2_B8A", "S2_B11", "S2_B12",
               "S2_NDVI", "S2_NDWI", "S2_SAVI", "S2_MNDWI", "S2_NBR"]

FULL_SPARSE_HLS30 = ["HLS_B2", "HLS_B3", "HLS_B4", "HLS_B5", 
                     "HLS_B6", "HLS_B7", "HLS_B9", "HLS_B10", "HLS_B11", "HLS_NDVI"]

# Training settings
HORIZONS = [7]              # predict * days ahead 
LOOKBACK = [7]                  # use past * days to predict next day
DEPTHS = [ 0.1 ]#, 0.4, 0.5] # 0.5]
        #    0.3, 0.4, 0.5]  # we will loop over these depths and train one model per depth
ALL_NETWORKS = [
    "all"]

    # ["COSMOS-UK", "GROW", "PTSMN", "TAHMO", "TERENO"]
SEPARATE_NETWORKS = [

    #  ["COSMOS-UK"],
            #  ["DWD"],
            #  ["FR_Aqui", "GROW"],
            #  ["PTSMN"], 
            # ["SMOSMANIA"], 
            # ["SOILSCAPE"],
            ["TAHMO","TERENO"],
        # ["TERENO"], 
            # ["TWENTE"], 
            # ["XMS-CAT"]
        ]

NETWORK1 = [
            ["TAHMO","TERENO","FR_Aqui"],
            ["TAHMO","TERENO","GROW"],
            ["TAHMO","TERENO","GROW","FR_Aqui"]
        ]


NB_WINDOWS = [100000]    
# Models: xgboost and lightgbm are fast, non-DL
MODELS = ["lstm"]
# , "lightgbm"]

def _without(lst, *items):
    return [x for x in lst if x not in items]

FEATURE_CONFIGS = [


    # # ── 4.2.1 Comparaison des modèles ──
    # {"name": "Models", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE,
    #  "lookbacks": [7], "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
    #  "models": ["xgboost", "lightgbm", "lstm", "gru", "tcn", "transformer"],
    #  "networks": ALL_NETWORKS},

#     # ── 4.2.2 Effet du lookback ──
#     {"name": "Lookback", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [1, 2, 4, 7, 14, 21, 28],
#      "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
#      "models": ["xgboost", "lstm"],
#      "networks": ALL_NETWORKS},

#     # ── 4.2.3 Effet de l'horizon ──
#     {"name": "Horizon", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [7],
#      "horizons": [1, 3, 7, 14, 21],
#      "depths": DEPTHS, "nb_windows": [50000],
#      "models": ["xgboost", "lstm"],
#      "networks": ALL_NETWORKS},

#     # ── 4.2.4 Effet du nombre de fenêtres ──
#     {"name": "Nb_Windows", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [7], "horizons": [7], "depths": DEPTHS,
#      "nb_windows": [1000, 5000, 10000, 20000, 50000, 100000],
#      "models": ["xgboost", "lstm"],
#      "networks": ALL_NETWORKS},

    {"name": "Network_FT", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE,
     "lookbacks": [7], "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
        "models": ["lstm"],
        "networks": ALL_NETWORKS},

#     # ── 4.2.6 Importance de la proximité géographique (par réseau) ──
#     {"name": "Networks", "dense": FULL_DENSE, "soil": FULL_SOIL, "sparse": FULL_SPARSE_S1,
#      "lookbacks": [7], "horizons": [7], "depths": DEPTHS, "nb_windows": [50000],
#      "models": ["xgboost"],
#      "networks": SEPARATE_NETWORKS},
]

SAVE_PLOTS = True
SAVE_NETWORKS_DIR = True
SAVE_MODELS_DIR = True
SAVE_RESULTS_CSV = True



TARGET_COL = "soil_moisture"
DATE_COL = "date_time"  # L'index temporel est sauvegardé sous date_time par process_timeseries
EPOCHS = 150
BATCH_SIZE = 32
SEED = 8

if SAVE_MODELS_DIR:
    os.makedirs(drive_dir, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# MONTHS=[4,5,6,7,8,9]
MONTHS = None

In [5]:
from Fcn_Training import (full_training, 
                        get_osiris_data,
                        Get_Grandvillers_data,
                        prepare_era5_dataset,
                        osiris_train,
                        osiris_fine_tuning,
                        full_eval_grandvillers,
                        full_eval_osiris,
                        evaluate_on_probes
                        )

In [4]:
# # 1. Charger les données de test Grandvillers
# test_list_local = Get_Grandvillers_data(os.path.join(Grandvillers_path, "Grandvillers_satellites"))
# # 2. Préparer directement la version de test ERA5 pour chaque fichier
# test_list_era5 = []
# print("--- PRÉPARATION DU JEU DE TEST ERA5 ---")
# for i, df in enumerate(test_list_local):
#     df_substituted = prepare_era5_dataset(df, display_name=f"Fichier {i}")
#     test_list_era5.append(df_substituted)

In [7]:
all_dfs = get_osiris_data(OSIRIS_DIR)

In [8]:
for feat_cfg in FEATURE_CONFIGS:
    feature_cols = feat_cfg["dense"] + feat_cfg["soil"] + feat_cfg["sparse"]
    osiris_train(feat_cfg, drive_dir, all_dfs)
    # full_eval_grandvillers(feat_cfg, drive_dir, test_list_local, test_list_era5)

Unique site_ids: ['2024_Champ1', '2024_Champ2', '2025_Cressonsacq', '2025_Grandvillers', '2025_Tarteron']

LEAVE-ONE-FIELD-OUT: test=2024_Champ1, val=2024_Champ2, train=['2025_Cressonsacq', '2025_Grandvillers', '2025_Tarteron']
Leave-one-out: test=2024_Champ1 (7 files), val=2024_Champ2 (7 files), train=16 files
--- Lookback: 7 ---
--- Horizon: 7 ---

=== Training with NB=50000 windows ===

      > Modèle: lstm
Train windows: 1244, Val windows: 511
Samples: train=1244, val=511  (lookback=7)
Epoch 1/150
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 0.5613 - val_loss: 2.3377 - learning_rate: 3.0000e-04
Epoch 2/150
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4555 - val_loss: 2.3109 - learning_rate: 3.0000e-04
Epoch 3/150
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.4075 - val_loss: 2.2954 - learning_rate: 3.0000e-04
Epoch 4/150
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.3804 - val_loss: 2.2902 - learning_rate: 3.0000e-04
Epoch 5/150
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

[TEST File 0 ] RMSE=[0.03322526 0.02976244 0.03181156 0.03284857 0.03073124 0.02987134
 0.02921021] ubRMSE=[0.01059632 0.0074678  0.00887483 0.00864268 0.00875269 0.01067501
 0.00890073] SMAPE=[12.44803071 11.32027954 12.06302345 12.55386323 11.64746806 11.0233821
 11.01676673] R²=[-13.93194008 -11.03341579 -12.11163807 -11.99657631  -9.54860878
  -8.15826607  -6.86852598] KGE=[-0.24726     0.00833911 -0.25342742 -0.29688935 -0.12973298 -0.63869094
 -0.16901858] Corr=[-0.24164877  0.09321302 -0.19362556 -0.17908699 -0.01258887 -0.53799967
  0.11598625]
✓ Résultat mis en tampon: FILE_0 - rmse = 0.033225,0.029762,0.031812,0.032849,0.030731,0.029871,0.029210
✓ Résultat mis en tampon: FILE_0 - ubrmse = 0.010596,0.007468,0.008875,0.008643,0.008753,0.010675,0.008901
✓ Résultat mis en tampon: FILE_0 - smape = 12.45,11.32,12.06,12.55,11.65,11.02,11.02
✓ Résultat mis en tampon: FILE_0 - r2 = -13.9319,-11.0334,-12.1116,-11.9966,-9.5486,-8.1583,-6.8685
✓ Résultat mis en tampon: FILE_0 - kge = -0.

[TEST File 0 ] RMSE=[0.04649016 0.03530582 0.03741623 0.0378161  0.0448475  0.0384859
 0.03880116] ubRMSE=[0.00944085 0.0073098  0.01092563 0.00914676 0.00734663 0.00921767
 0.00656935] SMAPE=[18.49924326 13.75455856 14.29620832 14.69988376 18.06035787 15.08128643
 15.50194174] R²=[-28.23488617 -15.93341637 -17.13874245 -16.22461319 -21.46526718
 -14.20223141 -12.8839674 ] KGE=[-0.12460606  0.37566374 -0.43250514 -0.08280955  0.39033634  0.1102443
  0.40337349] Corr=[-0.10723598  0.38915384 -0.42068402 -0.03535875  0.45580435  0.17350272
  0.70257878]
✓ Résultat mis en tampon: FILE_0 - rmse = 0.046490,0.035306,0.037416,0.037816,0.044847,0.038486,0.038801
✓ Résultat mis en tampon: FILE_0 - ubrmse = 0.009441,0.007310,0.010926,0.009147,0.007347,0.009218,0.006569
✓ Résultat mis en tampon: FILE_0 - smape = 18.50,13.75,14.30,14.70,18.06,15.08,15.50
✓ Résultat mis en tampon: FILE_0 - r2 = -28.2349,-15.9334,-17.1387,-16.2246,-21.4653,-14.2022,-12.8840
✓ Résultat mis en tampon: FILE_0 - kge = -

In [8]:
### Training
for feat_cfg in FEATURE_CONFIGS:
    # Réassigner les globales
    feature_cols = feat_cfg["dense"] + feat_cfg["soil"] + feat_cfg["sparse"]
    print(f"\n========== FEATURE SET: {feat_cfg['name']} ==========")
    print(f"  Features: {feature_cols}")

    ############### Training ISMN ###############
    # full_training(feat_cfg, base_path, drive_dir, MONTHS)

    # ########### Evaluation on Osiris ###############
    full_eval_osiris(feat_cfg, drive_dir, all_dfs)
    # evaluate_on_probes(feat_cfg, drive_dir, all_dfs)

    # ########## Evaluation on Grandvillers ###############
    # full_eval_grandvillers(feat_cfg, drive_dir, test_list_local, test_list_era5)

    # ############ Fine-Tuning Osiris ###############
    # osiris_fine_tuning(feat_cfg, drive_dir, all_dfs)

    # ########## Evaluation on Grandvillers ###############
    # for i in range(len(feat_cfg["models"])):
    #     feat_cfg["models"][i] = feat_cfg["models"][i] + "_fine_tuned"
    # feat_cfg["name"] = feat_cfg["name"] + "_fine_tuned"
    # full_eval_grandvillers(feat_cfg, drive_dir, test_list_local, test_list_era5)


========== FEATURE SET: Network_FT ==========
  Features: ['ET0', 'IRRAD', 'TMIN', 'TMAX', 'VAP', 'WIND', 'RAIN', 'VPD', 'T_RANGE', 'RAIN_CUM_3D', 'RAIN_CUM_7D', 'RAIN_CUM_14D', 'doy_sin', 'doy_cos', 'clay', 'silt', 'bulk', 'sand', 'dem', 'ksat_m_1km', 'dem_slope', 'dem_aspect', 'dem_twi', 'S2_B2', 'S2_B3', 'S2_B4', 'S2_B5', 'S2_B6', 'S2_B7', 'S2_B8', 'S2_B8A', 'S2_B11', 'S2_B12', 'S2_NDVI', 'S2_NDWI', 'S2_SAVI', 'S2_MNDWI', 'S2_NBR', 'S1_VV', 'S1_VH', 'S1_angle', 'S1_VV_over_VH', 'S1_VH_over_VV', 'HLS_B2', 'HLS_B3', 'HLS_B4', 'HLS_B5', 'HLS_B6', 'HLS_B7', 'HLS_B9', 'HLS_B10', 'HLS_B11', 'HLS_NDVI']

Evaluation Osiris: D=0.1, LB=7, H=7, NB=50000, Modèle=lstm
Dossier du bloc exploité:  -> /content/gdrive/MyDrive/Soil_Moisture/outputs/Fine_Tuning_Osiris/features_1/depth_0.1/lookback_7/horizon_7/nbwindows_50000/model_lstm/network_all

Skipping 2024_Champ1_887587: missing sparse features.
Skipping 2024_Champ1_887717: missing sparse features.
Skipping 2024_Champ1_891481: missing sparse fea